### Libraries

In [1]:
from anndata import AnnData
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import scanpy as sc
import torch

from sc_exp_design.config import NeuralVelocityFieldConfig
from sc_exp_design.models import FlowMatching

# defining figure size
FIGSIZE = (3, 3)
rcParams["figure.figsize"] = FIGSIZE


### Reading data

In [2]:
PATH = "../../theis/HID01_fcs_concatenated.h5ad"
adata = sc.read_h5ad(PATH)
adata


AnnData object with n_obs × n_vars = 93278696 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'experiment_number', 'experiment_id', 'mtg_[µm]', 'rhflt3l_[ng/ml]', 'gm-csf_[ng/ml]', 'tpo_[ng/ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng/ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng/ml]', 'il3_[ng/ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch'
    var: 'n', 'channel', 'marker', 'PnD', 'PnB', 'PnR', 'PnG', 'PnE', 'signal_type', 'antibody'
    uns: 'meta'

### Subsampling

In [3]:
# optionally subsample the data
KEEP_COPY = True
SUBSAMPLE = True
SUBSAMPLING_DONE = False
SUBSAMPLE_SIZE = 1e-2

if SUBSAMPLE and not SUBSAMPLING_DONE:
    if KEEP_COPY:
        adata_original = adata.copy()
        sc.pp.subsample(adata, SUBSAMPLE_SIZE)
    SUBSAMPLING_DONE = True
adata


AnnData object with n_obs × n_vars = 932786 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'experiment_number', 'experiment_id', 'mtg_[µm]', 'rhflt3l_[ng/ml]', 'gm-csf_[ng/ml]', 'tpo_[ng/ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng/ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng/ml]', 'il3_[ng/ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch'
    var: 'n', 'channel', 'marker', 'PnD', 'PnB', 'PnR', 'PnG', 'PnE', 'signal_type', 'antibody'
    uns: 'meta'

### Applying transformation

In [4]:
cofactor = 100
key_arcsin = f"X_arcsinh_cof{cofactor}" 
adata.obsm[key_arcsin] = np.arcsinh(adata.X/cofactor)
adata


AnnData object with n_obs × n_vars = 932786 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'experiment_number', 'experiment_id', 'mtg_[µm]', 'rhflt3l_[ng/ml]', 'gm-csf_[ng/ml]', 'tpo_[ng/ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng/ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng/ml]', 'il3_[ng/ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch'
    var: 'n', 'channel', 'marker', 'PnD', 'PnB', 'PnR', 'PnG', 'PnE', 'signal_type', 'antibody'
    uns: 'meta'
    obsm: 'X_arcsinh_cof100'

### Handling conditions

In [10]:
EXPERIMENTAL_COLUMNS = [
    'mtg_[µm]', 'rhflt3l_[ng/ml]', 'gm-csf_[ng/ml]', 'tpo_[ng/ml]', 'sr1_[nm]',
    'um171_[nm]', 'um729_[µm]', 'scf_[ng/ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]',
    'ldl_[ng/ml]', 'il3_[ng/ml]', 'o2_[%]',
]
unique_conds = list(adata_original.obs.groupby(by=EXPERIMENTAL_COLUMNS).groups.keys())
len(unique_conds)


/tmp/ipykernel_2519433/3500847691.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  unique_conds = list(adata_original.obs.groupby(by=EXPERIMENTAL_COLUMNS).groups.keys())


73

### One hot encode conditions.

In [ ]:
unique_conds


[(100, 10, '0', 0.0, 0, 0.0, 1.5, 0, 0, 0, 0, 0.0, 20),
 (100, 10, '0', 0.0, 0, 0.0, 1.5, 0, 100, 0, 0, 0.0, 20),
 (100, 10, '0', 0.0, 0, 70.0, 0.0, 0, 100, 0, 0, 0.0, 20),
 (100, 10, '0', 0.0, 0, 1120.0, 0.0, 0, 0, 0, 0, 0.0, 20),
 (100, 10, '0', 0.25, 0, 0.0, 1.5, 0, 0, 0, 0, 0.0, 20),
 (100, 10, '0', 0.5, 0, 0.0, 1.5, 0, 0, 0, 0, 0.0, 20),
 (100, 10, '0', 1.0, 0, 0.0, 1.5, 0, 0, 0, 0, 0.0, 20),
 (100, 10, '0', 2.5, 0, 0.0, 1.5, 0, 0, 0, 0, 0.0, 20),
 (100, 10, '1', 0.0, 0, 0.0, 1.5, 0, 100, 0, 0, 0.0, 20),
 (100, 10, '1', 0.0, 0, 280.0, 0.0, 0, 100, 0, 0, 0.0, 20),
 (100, 10, '1', 0.0, 75, 0.0, 1.5, 0, 100, 0, 0, 0.0, 20),
 (100, 10, '1', 0.0, 75, 280.0, 0.0, 0, 100, 0, 0, 0.0, 20),
 (100, 10, '1', 0.0, 375, 0.0, 1.5, 0, 100, 0, 0, 0.0, 20),
 (100, 10, '1', 0.0, 375, 280.0, 0.0, 0, 100, 0, 0, 0.0, 20),
 (100, 10, '10', 0.0, 0, 0.0, 1.5, 0, 100, 0, 0, 0.0, 20),
 (100, 10, '10', 0.0, 0, 280.0, 0.0, 0, 100, 0, 0, 0.0, 20),
 (100, 10, '10', 0.0, 75, 0.0, 1.5, 0, 100, 0, 0, 0.0, 20),
 (1